# Exploratory Data Analysis (EDA)
## Parkinson's Disease Detection — UCI Oxford Dataset
**MCA Final Year Project** · Shivam Kothiyal · H.N.B. Garhwal University

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='Greens_r')
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv(os.path.join('..', 'data', 'raw', 'parkinsons.data'))
print('Shape:', df.shape)
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Basic Info ===')
print(f'Total samples : {len(df)}')
print(f'Features      : {df.shape[1] - 2}  (excluding name and status)')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print('=== Class Distribution ===')
vc = df['status'].value_counts()
print(f'Parkinson\'s (1): {vc[1]}  ({vc[1]/len(df)*100:.1f}%)')
print(f'Healthy     (0): {vc[0]}  ({vc[0]/len(df)*100:.1f}%)')

## 2. Class Imbalance Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
labels = ["Parkinson's", 'Healthy']
values = [147, 48]
colors = ['#DC2626', '#16A34A']
axes[0].bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Dataset Class Imbalance (75.4% PD vs 24.6% Healthy)',
             fontsize=11, style='italic', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Distributions — PD vs Healthy

In [ ]:
key_features = ['PPE', 'RPDE', 'DFA', 'spread1', 'HNR',
                 'MDVP:Jitter(%)', 'MDVP:Shimmer', 'NHR']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    pd_vals  = df[df['status'] == 1][feat]
    hlt_vals = df[df['status'] == 0][feat]
    axes[i].hist(pd_vals,  bins=20, alpha=0.7, color='#DC2626', label="Parkinson's")
    axes[i].hist(hlt_vals, bins=20, alpha=0.7, color='#16A34A', label='Healthy')
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions: Parkinson\'s vs Healthy',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('Note: PPE and RPDE show the strongest separation between classes.')

## 4. Correlation Heatmap

In [ ]:
features_only = df.drop(['name', 'status'], axis=1)
corr = features_only.corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
            annot=False, linewidths=0.5, fmt='.2f',
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()
print('High correlation groups: Jitter features, Shimmer features — expected as they measure related acoustic properties.')

## 5. Feature Correlation with Target (status)

In [ ]:
target_corr = df.drop(['name'], axis=1).corr()['status'].drop('status').sort_values()

colors = ['#DC2626' if v > 0 else '#16A34A' for v in target_corr.values]
plt.figure(figsize=(10, 8))
plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Parkinson\'s Diagnosis (status)',
          fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()
print('PPE has the highest positive correlation with Parkinson\'s Disease — confirming it is the most discriminative feature.')

## 6. Boxplots — Top 6 Discriminative Features

In [ ]:
top_feats = ['PPE', 'spread1', 'RPDE', 'DFA', 'HNR', 'NHR']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    data_pd  = df[df['status'] == 1][feat]
    data_hlt = df[df['status'] == 0][feat]
    axes[i].boxplot([data_hlt, data_pd],
                    labels=['Healthy', "Parkinson's"],
                    patch_artist=True,
                    boxprops=dict(facecolor='#DCFCE7'),
                    medianprops=dict(color='#16A34A', linewidth=2))
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Value')

plt.suptitle('Boxplots — Top Discriminative Features',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Pairplot — Key Features

In [ ]:
plot_feats = ['PPE', 'RPDE', 'spread1', 'HNR', 'status']
pair_df = df[plot_feats].copy()
pair_df['Diagnosis'] = pair_df['status'].map({1: "Parkinson's", 0: 'Healthy'})

g = sns.pairplot(pair_df.drop('status', axis=1), hue='Diagnosis',
                  palette={"Parkinson's": '#DC2626', 'Healthy': '#16A34A'},
                  diag_kind='kde', plot_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot of Top 4 Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

## 8. Statistical Summary — PD vs Healthy

In [ ]:
pd_group  = df[df['status'] == 1].drop(['name','status'], axis=1)
hlt_group = df[df['status'] == 0].drop(['name','status'], axis=1)

summary = pd.DataFrame({
    'PD Mean'     : pd_group.mean().round(5),
    'Healthy Mean': hlt_group.mean().round(5),
    'Difference'  : (pd_group.mean() - hlt_group.mean()).round(5),
    'PD Std'      : pd_group.std().round(5),
})
summary['Separation'] = (abs(summary['Difference']) / summary['PD Std']).round(3)
summary.sort_values('Separation', ascending=False).head(10)

## Summary

| Finding | Detail |
|---|---|
| Best feature | **PPE** (Pitch Period Entropy) — highest correlation + separation |
| Class imbalance | 75.4% PD vs 24.6% Healthy — requires stratified split |
| Correlated groups | Jitter features, Shimmer features (high internal correlation) |
| Missing values | None — dataset is clean |
| Total features | 22 biomedical voice measurements |
